In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings
import numpy as np
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

from gridr.chain.grid_resampling_chain import basic_grid_resampling_chain
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
from chain_notebook_utils import (
    write_array, create_grid, create_star_polygon, plot_grid_on_image,
)

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# File paths shared across the chain tutorial series.
RASTER_IN          = "./grid_resampling_chain_raster_in.tif"
GRID_IN_F64        = "./grid_resampling_chain_grid_in_f64.tif"
output_raster_path = "./grid_resampling_chain_output_raster.tif"
output_mask_path   = "./grid_resampling_chain_output_mask.tif"

# Grid parameters shared across the chain tutorial series.
nrow, ncol = 50, 40
origin_pos  = np.array((0.3, 0.2))
origin_node = np.array((0., 0.))
v_row_y, v_row_x = 5.2, 1.2
v_col_y, v_col_x = -2.7, 7.1

grid_row_f64, grid_col_f64 = create_grid(
    nrow, ncol, origin_pos, origin_node,
    v_row_y, v_row_x, v_col_y, v_col_x, dtype=np.float64,
)

# Hybrid input creation: ensure the source raster and grid file exist on disk.
if not os.path.exists(RASTER_IN):
    write_array(mandrill, dtype=mandrill.dtype, fileout=RASTER_IN)
if not os.path.exists(GRID_IN_F64):
    write_array(np.array([grid_row_f64, grid_col_f64]),
                dtype=np.float64, fileout=GRID_IN_F64)

# Default output-dataset open arguments (resolution-dependent overrides
# are applied per-notebook below).
grid_resolution = (8, 8)
from gridr.core.grid.grid_commons import grid_full_resolution_shape
output_shape = grid_full_resolution_shape(
    shape=grid_row_f64.shape, resolution=grid_resolution,
)
raster_out_open_args = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape[0], "width": output_shape[1], "count": 1,
}
mask_out_open_args = {
    "driver": "GTiff", "dtype": np.uint8,
    "height": output_shape[0], "width": output_shape[1],
    "count": 1, "nbits": 1,
}

grid_mask_in_path        = "./grid_resampling_chain_grid_mask_in_u8_1_0.tif"
grid_mask_in_path_i8     = "./grid_resampling_chain_grid_mask_in_i8_0_m10.tif"
raster_mask_in_path_u8   = "./grid_resampling_chain_raster_mask_in_u8_1_0.tif"

# Ensure the grid mask exists on disk (created in notebook 003 if missing).
grid_mask_in_valid_value, grid_mask_in_invalid_value = 1, 0
roi = np.array(((10, 40), (5, 100)))
grid_mask = np.full(grid_row_f64.shape, grid_mask_in_valid_value, dtype=np.uint8)
grid_mask[
    np.logical_and(
        np.logical_and(grid_row_f64 >= roi[0][0], grid_row_f64 <= roi[0][1]),
        np.logical_and(grid_col_f64 >= roi[1][0], grid_col_f64 <= roi[1][1]),
    )
] = grid_mask_in_invalid_value
if not os.path.exists(grid_mask_in_path):
    write_array(grid_mask, dtype=np.uint8, fileout=grid_mask_in_path)

# Ensure the source raster mask exists on disk (created in notebook 004 if missing).
array_src_mask_validity_valid, array_src_mask_validity_invalid = 1, 0
array_in_mask = np.full(mandrill[0].shape, array_src_mask_validity_valid, dtype=np.uint8)
array_in_mask[slice(50, 71), slice(150, 171)] = array_src_mask_validity_invalid
if not os.path.exists(raster_mask_in_path_u8):
    write_array(array_in_mask, dtype=np.uint8, fileout=raster_mask_in_path_u8)


# Source raster mask

A *source raster mask* flags pixels in the source image as invalid. Any output sample whose interpolation footprint overlaps an invalid source pixel is itself flagged invalid and set to `nodata_out`. Unlike the grid mask, which operates on output samples, the source raster mask operates on input pixels.

**What you'll learn**

- How to build and save a source raster mask
- How to pass it through `array_src_mask_ds`,
  `array_src_mask_validity_pair` and `array_src_mask_band`
- How source-mask invalidation propagates to the output mask

## Setting things up

We reuse the source raster, grid file and grid mask from the previous notebook. The source raster mask itself is created below if absent.

## Building the source raster mask

We mark a small square near the mandrill's left eye as invalid.

In [ ]:
array_src_mask_validity_valid, array_src_mask_validity_invalid = 1, 0

# Create the mask raster, initially all valid.
array_in_mask = np.full(mandrill[0].shape, array_src_mask_validity_valid, dtype=np.uint8)
array_in_mask[slice(50, 71), slice(150, 171)] = array_src_mask_validity_invalid

write_array(array_in_mask, dtype=np.uint8, fileout=raster_mask_in_path_u8)

The invalid region is shown below as a red square overlaid on the source image. The grid mask coloured nodes are also visible.

In [ ]:
plot_grid_on_image(
    1.5, grid_row_f64, grid_col_f64, (10, 10),
    (mandrill.shape[1], mandrill.shape[2]),
    mask=grid_mask, win=None, raster_image=mandrill[0],
    raster_image_mask=array_in_mask,
    prefix="array_in_mask_input",
)

## Running the chain with the source mask

Three arguments are needed:
- `array_src_mask_ds` -- opened mask dataset,
- `array_src_mask_validity_pair` -- `(valid_value, invalid_value)` tuple,
- `array_src_mask_band` -- band index to read.

In [ ]:
with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(grid_mask_in_path, "r") as grid_mask_in_ds, \
        rasterio.open(raster_mask_in_path_u8, "r") as raster_mask_in_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds                        = grid_in_ds,
        grid_row_coords_band           = 1,
        grid_col_coords_band           = 2,
        grid_resolution                = grid_resolution,
        array_src_ds                   = array_src_ds,
        array_src_bands                = 1,
        array_out_ds                   = array_out_ds,
        interp                         = "cubic",
        nodata_out                     = 0,
        mask_out_ds                    = mask_out_ds,
        grid_mask_in_ds                = grid_mask_in_ds,
        grid_mask_in_unmasked_value    = grid_mask_in_valid_value,
        grid_mask_in_band              = 1,
        array_src_mask_ds              = raster_mask_in_ds,
        array_src_mask_band            = 1,
        array_src_mask_validity_pair   = (array_src_mask_validity_valid,
                                          array_src_mask_validity_invalid),
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as raster_ds, \
        rasterio.open(output_mask_path, "r") as mask_ds:
    plot_im({"output image": raster_ds.read(1), "output mask": mask_ds.read(1)},
            prefix="array_in_mask_output")

Both masks contribute to the output mask: any output sample is invalid if its grid node is masked **or** if its interpolation kernel touches an invalid source pixel.